In [ ]:
import akshare as ak
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from datetime import datetime, timedelta

# -------------------------- 1. 初始化参数（修正股票代码格式） --------------------------
# akshare的A股代码格式："600636"（无需加sh/sz，接口自动识别）
stock_codes = [
    "600636",  # 目标股：*ST国化
    "002230",  # 科大讯飞（教育信息化龙头）
    "002841",  # 视源股份（教育设备）
    "000001",  # 上证指数（市场整体趋势）
    "399006"   # 创业板指（成长板块参考）
]

# 数据时间范围（建议至少3年，可调整）
start_date = "2020-01-01"
end_date = datetime.now().strftime("%Y-%m-%d")  # 今日日期

# -------------------------- 2. 批量获取股票/指数历史数据（修正字段问题） --------------------------
def get_stock_data(code, start, end):
    """
    获取单只股票/指数的日线数据
    akshare返回的字段：date(日期), open(开盘价), close(收盘价), high(最高价), low(最低价), volume(成交量), ...
    确保字段名称准确，避免KeyError
    """
    try:
        # 判断是股票还是指数（指数代码以399/000开头）
        if code.startswith(("000", "399")):
            # 获取指数数据
            df = ak.stock_zh_index_hist(symbol=code, period="daily", start_date=start, end_date=end)
        else:
            # 获取A股股票数据（后复权，消除除权除息影响）
            df = ak.stock_zh_a_hist(symbol=code, period="daily", start_date=start, end_date=end, adjust="qfq")

        # 保留需要的字段（确保字段存在）
        required_cols = ["date", "close", "volume"]
        # 计算5日均线和10日均线（如果接口没返回，手动计算）
        df["ma5"] = df["close"].rolling(window=5).mean()
        df["ma10"] = df["close"].rolling(window=10).mean()

        # 只保留必需字段
        df = df[required_cols + ["ma5", "ma10"]]
        return df
    except Exception as e:
        print(f"❌ 获取 {code} 数据失败：{str(e)}")
        return pd.DataFrame()

# 批量获取所有股票/指数数据
stock_data = {}
for code in stock_codes:
    df = get_stock_data(code, start_date, end_date)
    if not df.empty:
        stock_data[code] = df
        print(f"✅ 已获取 {code} 数据：{len(df)} 条记录")
    else:
        print(f"❌ {code} 数据获取失败，跳过")

# 检查是否成功获取目标股数据
target_code = "600636"
if target_code not in stock_data:
    raise ValueError(f"目标股 {target_code} 数据获取失败，请检查网络或代码")

# -------------------------- 3. 数据预处理（对齐时间轴，合并特征） --------------------------
# 以目标股600636的日期为基准，合并其他股票/指数的收盘价
target_df = stock_data[target_code][["date", "close"]].rename(columns={"close": "target_close"})

# 合并其他股票/指数的收盘价作为特征
for code in stock_codes:
    if code != target_code and code in stock_data:
        feat_df = stock_data[code][["date", "close"]].rename(columns={"close": f"{code}_close"})
        target_df = pd.merge(target_df, feat_df, on="date", how="left")

# 处理缺失值（前向填充+删除剩余空行）
target_df = target_df.fillna(method="ffill").dropna()
print(f"\n合并后的数据形状：{target_df.shape}")
print("数据前5行：")
print(target_df.head())

# -------------------------- 4. 特征工程（时间序列滞后特征） --------------------------
lag_days = 3  # 用前3天数据预测第4天，可调整
# 目标股的滞后特征
for i in range(1, lag_days + 1):
    target_df[f"lag_{i}"] = target_df["target_close"].shift(i)

# 核心相关股（科大讯飞002230）的滞后特征（增强模型关联性）
if "002230_close" in target_df.columns:
    for i in range(1, lag_days + 1):
        target_df[f"002230_lag_{i}"] = target_df["002230_close"].shift(i)

# 删除滞后特征产生的空行
target_df = target_df.dropna()

# 定义特征X和目标y
X_cols = [col for col in target_df.columns if col not in ["date", "target_close"]]
X = target_df[X_cols].values
y = target_df["target_close"].values

# 按时间顺序划分训练集（80%）和测试集（20%）
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"\n训练集大小：{len(X_train)}，测试集大小：{len(X_test)}")
print(f"特征列：{X_cols}")

# -------------------------- 5. 模型训练与预测 --------------------------
# 初始化线性回归模型（入门级，稳定易理解）
model = LinearRegression()
model.fit(X_train, y_train)

# 测试集预测
y_pred = model.predict(X_test)

# 模型评估
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"\n模型评估结果：")
print(f"均方误差（MSE）：{mse:.4f}（越小越好）")
print(f"决定系数（R²）：{r2:.4f}（越接近1越好）")

# 预测未来1个交易日收盘价
latest_data = target_df.iloc[-1][X_cols].values.reshape(1, -1)
next_day_pred = model.predict(latest_data)[0]
last_date = target_df.iloc[-1]["date"]
print(f"\n📈 预测 {last_date} 下一个交易日收盘价：{next_day_pred:.2f} 元")

# -------------------------- 6. 数据可视化（中文正常显示） --------------------------
plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows中文支持
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题
plt.figure(figsize=(15, 10))

# 子图1：600636历史收盘价趋势
plt.subplot(2, 2, 1)
plt.plot(target_df["date"], target_df["target_close"], label="600636收盘价", color="#1f77b4")
plt.title("600636(*ST国化)历史收盘价趋势", fontsize=12, fontweight="bold")
plt.xlabel("日期")
plt.ylabel("收盘价（元）")
plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.3)

# 子图2：实际值vs预测值
plt.subplot(2, 2, 2)
train_dates = target_df["date"][:train_size]
test_dates = target_df["date"][train_size:]
plt.plot(train_dates, y_train, label="训练集实际值", color="#1f77b4", alpha=0.7)
plt.plot(test_dates, y_test, label="测试集实际值", color="#2ca02c")
plt.plot(test_dates, y_pred, label="测试集预测值", color="#ff7f0e", linestyle="--", linewidth=2)
plt.title("实际收盘价 vs 预测收盘价", fontsize=12, fontweight="bold")
plt.xlabel("日期")
plt.ylabel("收盘价（元）")
plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.3)

# 子图3：600636与科大讯飞相关性
if "002230_close" in target_df.columns:
    plt.subplot(2, 2, 3)
    plt.scatter(target_df["002230_close"], target_df["target_close"], alpha=0.5, color="#9467bd")
    # 添加趋势线
    z = np.polyfit(target_df["002230_close"], target_df["target_close"], 1)
    p = np.poly1d(z)
    plt.plot(target_df["002230_close"], p(target_df["002230_close"]), "r--", alpha=0.8)
    plt.title("600636与科大讯飞收盘价相关性", fontsize=12, fontweight="bold")
    plt.xlabel("科大讯飞收盘价（元）")
    plt.ylabel("600636收盘价（元）")
    plt.grid(alpha=0.3)

# 子图4：预测误差分布
plt.subplot(2, 2, 4)
errors = y_test - y_pred
plt.hist(errors, bins=20, color="#ff7f0e", alpha=0.7, edgecolor="black")
plt.axvline(x=0, color="red", linestyle="--", alpha=0.8, label="误差=0")
plt.title("预测误差分布", fontsize=12, fontweight="bold")
plt.xlabel("误差（实际值-预测值）")
plt.ylabel("频次")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# -------------------------- 7. 结果保存（Excel文件） --------------------------
result_df = target_df.copy()
# 新增预测列（训练集无预测值，测试集填充预测值）
result_df["predict_close"] = np.nan
result_df.iloc[train_size:, -1] = y_pred
# 保存到Excel
result_df.to_excel("600636_stock_prediction_fixed.xlsx", index=False)
print(f"\n📊 结果已保存到：600636_stock_prediction_fixed.xlsx")